### Ingest Audit Log CSV
This notebook ingests the raw audit log file from the landing layer, standardizes the columns, and writes the result to a bronze Delta table for downstream processing.

Business context:
* This audit log captures bidder activity during an auction
* It includes events such as proxy bids, outbids, logins, and related user actions
* The bronze table preserves these events so it can support later analysis of bidding behavior and auction activity

#### Step 1: Load shared setup values
This step imports the common project configuration used across bronze ingestion notebooks.

What happens here:
* Loads shared Unity Catalog and volume variables
* Reuses helper functions such as metadata column enrichment
* Keeps environment-specific values centralized in one setup notebook


#### Step 2: Build the audit log file path
This step assembles the full storage path for the raw audit log CSV.

What happens here:
* Combines the configured volume, catalog, schema, and folder names
* Points to the specific dated audit log file to ingest
* Makes the resolved source path easy to validate before reading data


#### Step 3: Define the audit log schema
This step declares the expected structure of the CSV so Spark reads each column with the intended type.

What happens here:
* Defines the incoming audit log column names explicitly
* Applies consistent types for identifiers, activity fields, and timestamps
* Avoids schema drift that can happen with CSV inference alone


#### Audit log data dictionary
This file represents auction activity events generated during the bidder journey.

Key fields:
* `Time of Bid Placed`: when the auction event happened
* `Item ID`: the item identifier associated with the event
* `Lot Number`: the auction lot tied to the event
* `Activity`: the event type, such as proxy bid, outbid, login, or other bidder actions
* `Bid price, User ID`: a combined source field containing bid-related or user-related details
* `From Bidder(IN) / To Clerk(OUT)`: indicates the event direction or participant flow captured by the source system

Business use:
* Track how bidders interact with auction lots over time
* Analyze bidding behavior and user activity patterns
* Support downstream reporting on auction engagement and event history


#### Step 4: Load and enrich the audit log data
This step reads the raw CSV into Spark and appends standard metadata columns used for traceability.

What happens here:
* Loads the audit log file into a Spark DataFrame
* Applies the predefined schema during ingestion
* Adds metadata such as source path and ingestion timestamp


#### Step 5: Prepare the bronze-ready dataset
This step finalizes the DataFrame for bronze storage by standardizing values and validating the target context.

What happens here:
* Converts the bid timestamp into a usable timestamp type
* Renames columns into a cleaner bronze-friendly naming standard
* Confirms the DataFrame is ready to be written to the bronze layer


#### Step 6: Write the bronze audit log table
This final step persists the cleaned audit log DataFrame as a Delta table in the bronze schema.

What happens here:
* Writes the transformed audit log data to a managed Delta table
* Rebuilds the target table on reruns using overwrite mode
* Makes the bronze audit log available for downstream silver transformations and analysis


In [ ]:
catalog = dbutils.widgets.get("catalog")
container = dbutils.widgets.get("container")
storage_account = dbutils.widgets.get("storage_account")

In [ ]:
%run "../setup/002_Environment_variables"

In [ ]:
dbutils.widgets.text('id'," ")
batch_id = dbutils.widgets.get('id')
batch_id

In [ ]:
# Build the full source path for the CSV file to ingest from cloud storage.
# This keeps the input file location parameterized through shared environment values.
path = f'/{volume}/{catalog}/{schema}/{file}/{batch_id}/audit_log_*.csv'

In [ ]:
dbutils.widgets.text('id'," ")
batch_id = dbutils.widgets.get('id')
batch_id

In [ ]:
# Import explicit Spark SQL data types so the CSV schema can be declared manually.
from pyspark.sql.types import *

# Define the expected structure of the source CSV.
# Using an explicit schema prevents unwanted type inference issues.
schema_auditlog = StructType([
        StructField('Time of Bid Placed', StringType(), True),
        StructField('Item ID', IntegerType(), True),
        StructField('Lot Number', IntegerType(), True),
        StructField('Activity', StringType(), True),
        StructField('Bid price, User ID', StringType(), True),
        StructField('From Bidder(IN) / To Clerk(OUT)', StringType(), True),
])

In [ ]:
# Read the source CSV into a Spark DataFrame using the predefined schema.
# The file is expected to contain a header row.
df_auditlog = spark.read.format('csv') \
    .option('header', 'true') \
         .option('multiLine', 'true') \
    .option("quote", '"') \
    .option("escape", '"') \
        .option('timestampFormat', 'yyyy-MM-dd HH:mm:ss z') \
    .schema(schema_auditlog) \
    .load(path)

# Add standard metadata or audit columns using the shared helper function.
df_auditlog = add_columns(df_auditlog)

In [ ]:
df_auditlog = df_auditlog.withColumn(
    "Time of Bid Placed",
    to_timestamp("Time of Bid Placed", "dd-MMM-yyyy HH:mm:ss.SSS")
)

In [ ]:
df_auditlog = df_auditlog.withColumnsRenamed({col: col.replace(', ', '_').replace(' / ', '_').replace(' ', '_').lower() for col in df_auditlog.columns})

df_auditlog = df_auditlog.withColumnsRenamed({'from_bidder(in)_to_clerk(out)': 'from_bidder_to_clerk'})

In [ ]:
# Write the ingested DataFrame to the bronze layer as a Delta table.
# Overwrite mode is used so the table can be rebuilt on repeated runs.
df_auditlog \
    .write \
    .format('delta') \
    .mode('overwrite') \
    .partitionBy('batch_id') \
    .option('replaceWhere',f"batch_id = '{batch_id}'")\
    .saveAsTable(f'{catalog}.{bronze}.auditlog')